In [1]:
import scanpy as sc
import anndata as ad
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd 

from pathlib import Path


# NO BAD, NOT M/A RATIO, RODS VS BIPOLARS

In [2]:
data_path = "/home/bmb/haxx/working/ceisel_mumm/data/"

In [3]:
micro_ratios = pd.read_csv(data_path + "mumm_microarray.csv",low_memory=False,header=[0,1],index_col=1)

micro_ratios.columns = pd.MultiIndex.from_arrays([
    micro_ratios.columns.get_level_values(0).to_series().replace(r"^Unnamed.*", pd.NA, regex=True).ffill(),
    micro_ratios.columns.get_level_values(1)
])

In [4]:
micro_ratios.head()

In [5]:
rod_ratios = micro_ratios['Rod'].copy()
rod_ratios['probeset_id'] = micro_ratios[(None,'probeset_id')].astype(str)
# rod_ratios.set_index('probeset_id')

rod_ratios.head()

In [6]:
# micro_ratios.loc['probeset_id']

In [7]:
from pybiomart import Server,Dataset

dataset = Dataset(name="drerio_gene_ensembl", host="http://useast.ensembl.org")


In [8]:
AFFY_ATTR = 'affy_zebrafish'

ATTRIBUTES = [
    AFFY_ATTR,
    "ensembl_gene_id",
    "ensembl_transcript_id",
    "external_gene_name",
    "transcript_biotype",
]

def get_full_mapping(dataset):
    print("Fetching full mapping table (this may take a minute)...")
    # Query without filters to get everything
    df = dataset.query(attributes=ATTRIBUTES)
    return df

mapping = get_full_mapping(dataset)

In [9]:
mapping.head()

In [10]:
inverted_mapping = {probe_id:str(ensembl_name) for ensembl_name,probe_id in zip(mapping['Gene name'],mapping['AFFY Zebrafish probe'])}

In [11]:
inverted_mapping

In [12]:
# all_probe_ids = [str(id) for id in micro_ratios[(None,'probeset_id')]]
all_probe_ids = rod_ratios['probeset_id']

available_mask = [
    id in inverted_mapping
    for id in all_probe_ids
]

In [15]:
available_ratios = rod_ratios[available_mask].copy()

In [16]:
available_ratios.head()

In [19]:
available_ratios['gene_name'] = available_ratios['probeset_id'].map(inverted_mapping)

In [20]:
filtered_ratios = available_ratios[available_ratios['gene_name'] != "nan"]

In [24]:
# Extract annotations 
header = available_ratios.columns.tolist()[1:-1]
annotations = {
    name:{
        "time":int(name.split("-")[1].strip("hr")),
        "replicate":name.split("-")[0],
        # "M/A": "M" if name.split(".")[-1] == "1" else "A",
        "column":np.array(available_ratios[name])
    }
    for name in header if name not in {'probeset_id','gene_name'}
}
annotations


In [ ]:
# join_ma = {}
# for name,v in annotations.items():
#     id = name.rstrip(".1")
#     column = v.pop('column')
#     m_a = v.pop('M/A')
#     join_ma[id] = v
#     if m_a == "M":
#         join_ma[id]['M'] = column
#     else:
#         join_ma[id]['A'] = column
# join_ma
        


In [ ]:
# for time in sorted(list({v['time'] for v in annotations.values()})):
#     time_columns = [c for c,v in annotations.items() if v['time'] == time]
#     print(time,time_columns)
#     print([annotations[c]['M/A'] for c in time_columns])

times = sorted(list({v['time'] for v in join_ma.values()}))
for time in times:
    print({k:v for k,v in join_ma.items() if v['time'] == time})

In [ ]:
available_ratios[['R1-8hr-MTZ.CEL vs R1-8hr-Dans.CEL', 'R1-8hr-MTZ.CEL vs R1-8hr-Dans.CEL.1',]].head()

In [ ]:
double_inverted = {
    v:k for k,v in inverted_mapping.items() if type(v) != float
}
double_inverted.pop("nan")

In [ ]:
[(g,g in double_inverted) for g in ['endog',
 'parp1',
 'il1b',
 'map2k1',
 'mapk1',
 'mmp9',
 'sirt1',
 'plat',
 'parga',
 'pargl',
 'dlg4a',
 'dlg4b',
 'dlg4b-1',
 'aif1l',
 'appa',
 'appb',
 'grin2bb']]

In [ ]:
double_inverted